# Vector Calculus — Interactive Visualisation Notebook
**PSE Lab · SeoulTech**

This notebook provides interactive visualisations for all six chapters of the Vector Calculus course.  
Run each cell to see the plots. Tweak the parameters and re-run to explore.

| Chapter | Topic | What you'll see |
|---|---|---|
| Ch 1 | Curves | Helix, curvature, Frenet frame |
| Ch 1 | Line integrals | Two paths, same endpoints, different results |
| Ch 2 | Area integrals | Jacobian distortion, polar coordinates |
| Ch 2 | Surface integrals | Sphere, flux arrows |
| Ch 3 | Grad / Div / Curl | Vector field operations, visualised |
| Ch 3 | Conservative vs non-conservative | Potential surfaces and circulation |
| Ch 4 | Divergence theorem | Volume vs surface flux comparison |
| Ch 4 | Stokes' theorem | Curl flux vs boundary circulation |
| Ch 5 | Laplace equation | 2D harmonic function, equipotential lines |
| Ch 5 | Poisson equation | Point charge potential |
| Ch 6 | Index notation | Levi-Civita cross product verification |

In [ ]:
# ── Setup: run this cell first ──────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib import cm
from mpl_toolkits.mplot3d import Axes3D
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
import warnings
warnings.filterwarnings('ignore')

# Consistent style
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor':   '#f8f8f6',
    'axes.grid':        True,
    'grid.alpha':       0.4,
    'axes.spines.top':  False,
    'axes.spines.right':False,
    'font.size':        11,
    'axes.titlesize':   13,
    'axes.titleweight': 'bold',
})
TEAL   = '#1D9E75'
PURPLE = '#7F77DD'
CORAL  = '#D85A30'
BLUE   = '#378ADD'
AMBER  = '#BA7517'
GRAY   = '#888780'
print('Setup complete. Ready to visualise!')

---
## Chapter 1 — Curves
### 1.1 The Helix: parametrisation, tangent vector, arc length

In [ ]:
# ── Ch1: Helix and Frenet frame ─────────────────────────────────────────────
# PARAMETERS — try changing these!
lam = 1.0      # speed parameter (higher = faster winding)
turns = 2      # number of full rotations to draw
frame_t = 2.5  # parameter value where Frenet frame is shown

t = np.linspace(0, 2 * np.pi * turns, 800)
x = np.cos(lam * t)
y = np.sin(lam * t)
z = lam * t

# Arc length: ds/dt = |x_dot| = sqrt(lam^2 + lam^2 + lam^2) = lam*sqrt(2)
speed = np.sqrt(lam**2 * np.sin(lam*t)**2 + lam**2 * np.cos(lam*t)**2 + lam**2)
arc_len = np.cumsum(speed) * (t[1] - t[0])

# Frenet frame at frame_t
def helix(t): return np.array([np.cos(lam*t), np.sin(lam*t), lam*t])
def helix_d(t): return np.array([-lam*np.sin(lam*t), lam*np.cos(lam*t), lam])
def helix_dd(t): return np.array([-lam**2*np.cos(lam*t), -lam**2*np.sin(lam*t), 0.0])

P  = helix(frame_t)
T  = helix_d(frame_t);   T  /= np.linalg.norm(T)
dd = helix_dd(frame_t);  N  = dd / np.linalg.norm(dd)
B  = np.cross(T, N)
scale = 0.7

fig = plt.figure(figsize=(14, 5))
ax1 = fig.add_subplot(1, 3, 1, projection='3d')
ax1.plot(x, y, z, color=TEAL, lw=2, label='helix')
ax1.quiver(*P, *(scale*T), color=CORAL,  arrow_length_ratio=0.2, lw=2, label='t (tangent)')
ax1.quiver(*P, *(scale*N), color=PURPLE, arrow_length_ratio=0.2, lw=2, label='n (normal)')
ax1.quiver(*P, *(scale*B), color=AMBER,  arrow_length_ratio=0.2, lw=2, label='b (binormal)')
ax1.scatter(*P, s=60, c='black', zorder=5)
ax1.set_title(f'Helix  (λ={lam})')
ax1.legend(fontsize=9, loc='upper left')
ax1.set_xlabel('x'); ax1.set_ylabel('y'); ax1.set_zlabel('z')

# Arc length vs t
ax2 = fig.add_subplot(1, 3, 2)
ax2.plot(t, arc_len, color=TEAL, lw=2)
ax2.set_xlabel('parameter t')
ax2.set_ylabel('arc length s')
ax2.set_title(f's = √2·λ·t  (slope = {lam*np.sqrt(2):.3f})')
ax2.axvline(frame_t, color=GRAY, ls='--', lw=1, label=f't = {frame_t}')
ax2.legend()

# Curvature of helix (constant = lam / (1 + lam^2) but shown vs s)
kappa = lam**2 / (lam**2 * 2)   # = 1/2 for lam=1
ax3 = fig.add_subplot(1, 3, 3)
ax3.axhline(kappa, color=PURPLE, lw=2, label=f'κ = λ/(1+λ²) = {kappa:.3f}')
ax3.axhline(0,     color=GRAY,   lw=1, ls='--', label='straight line (κ=0)')
ax3.set_xlabel('arc length s')
ax3.set_ylabel('curvature κ')
ax3.set_title('Curvature is constant for a helix')
ax3.set_ylim(-0.1, 1.0)
ax3.legend()

plt.suptitle('Ch 1: The Helix — Frenet Frame, Arc Length, Curvature', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()
print(f'Arc length for {turns} full turns = {arc_len[-1]:.4f}  (theory: {2*np.pi*turns*np.sqrt(2)*lam:.4f})')

### 1.2 Line integrals — two paths, same endpoints

In [ ]:
# ── Ch1: Line integrals — path dependence ───────────────────────────────────
# F(x,y,z) = (x*e^y, z^2, x*y)  from (0,0,0) to (1,1,1)
# C1: x(t) = (t, t^2, t^3)     -> integral = (1 + 2e)/4
# C2: x(t) = (t, t, t)         -> integral = 5/3

t = np.linspace(0, 1, 400)

# C1
x1, y1, z1 = t, t**2, t**3
dx1, dy1, dz1 = np.ones_like(t), 2*t, 3*t**2
F1x = x1 * np.exp(y1);  F1y = z1**2;  F1z = x1 * y1
integrand1 = F1x*dx1 + F1y*dy1 + F1z*dz1
val1 = np.trapz(integrand1, t)

# C2
x2, y2, z2 = t, t, t
dx2, dy2, dz2 = np.ones_like(t), np.ones_like(t), np.ones_like(t)
F2x = x2 * np.exp(y2);  F2y = z2**2;  F2z = x2 * y2
integrand2 = F2x*dx2 + F2y*dy2 + F2z*dz2
val2 = np.trapz(integrand2, t)

exact1 = (1 + 2*np.e) / 4
exact2 = 5 / 3

fig = plt.figure(figsize=(13, 5))
ax1 = fig.add_subplot(1, 2, 1, projection='3d')
ax1.plot(x1, y1, z1, color=TEAL,   lw=3, label=f'C₁: (t, t², t³)\n∫={val1:.4f}  (exact {exact1:.4f})')
ax1.plot(x2, y2, z2, color=CORAL,  lw=3, label=f'C₂: (t, t, t)\n∫={val2:.4f}  (exact {exact2:.4f})')
ax1.scatter([0,1],[0,1],[0,1], s=80, c='black', zorder=5)
ax1.text(0, 0, 0, ' (0,0,0)', fontsize=9)
ax1.text(1, 1, 1, ' (1,1,1)', fontsize=9)
ax1.set_xlabel('x'); ax1.set_ylabel('y'); ax1.set_zlabel('z')
ax1.set_title('Two paths between same endpoints')
ax1.legend(fontsize=9)

# Running integrals
ax2 = fig.add_subplot(1, 2, 2)
run1 = np.array([np.trapz(integrand1[:k+1], t[:k+1]) for k in range(len(t))])
run2 = np.array([np.trapz(integrand2[:k+1], t[:k+1]) for k in range(len(t))])
ax2.plot(t, run1, color=TEAL,  lw=2, label=f'C₁  →  {val1:.4f}')
ax2.plot(t, run2, color=CORAL, lw=2, label=f'C₂  →  {val2:.4f}')
ax2.axhline(exact1, color=TEAL,  ls='--', lw=1, alpha=0.6)
ax2.axhline(exact2, color=CORAL, ls='--', lw=1, alpha=0.6)
ax2.set_xlabel('parameter t')
ax2.set_ylabel('cumulative ∫F·dx')
ax2.set_title('Running totals diverge — path matters!')
ax2.legend()

plt.suptitle('Ch 1: Line Integrals — F = (xeʸ, z², xy)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()
print(f'Difference: {abs(val1-val2):.4f}  (the field is NOT conservative)')

---
## Chapter 2 — Surfaces and Volumes
### 2.1 Jacobian — how coordinate changes distort area

In [ ]:
# ── Ch2: Jacobian — Cartesian grid vs polar grid ─────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Left: Cartesian grid
ax = axes[0]
for xi in np.linspace(0, 2, 9):
    ax.axvline(xi, color=TEAL, lw=0.8, alpha=0.7)
for yi in np.linspace(0, 2, 9):
    ax.axhline(yi, color=TEAL, lw=0.8, alpha=0.7)
# highlight one small rectangle
dx, dy = 0.25, 0.25
rx, ry = 0.75, 0.75
rect = plt.Rectangle((rx, ry), dx, dy, fc=TEAL, alpha=0.4, ec=TEAL, lw=2)
ax.add_patch(rect)
ax.text(rx + dx/2, ry + dy/2, f'δA = δx·δy\n= {dx*dy:.4f}',
        ha='center', va='center', fontsize=10, fontweight='bold')
ax.set_xlim(-0.1, 2.1); ax.set_ylim(-0.1, 2.1)
ax.set_aspect('equal'); ax.set_xlabel('x'); ax.set_ylabel('y')
ax.set_title('Cartesian grid (J = 1)')

# Right: polar grid
ax = axes[1]
rhos = np.linspace(0.2, 2.0, 8)
phis = np.linspace(0, np.pi/2, 10)
for r in rhos:
    ph = np.linspace(0, np.pi/2, 200)
    ax.plot(r*np.cos(ph), r*np.sin(ph), color=CORAL, lw=0.8, alpha=0.7)
for ph in phis:
    rr = np.linspace(0.2, 2.0, 100)
    ax.plot(rr*np.cos(ph), rr*np.sin(ph), color=CORAL, lw=0.8, alpha=0.7)
# highlight one polar cell
r0, p0 = 1.0, np.pi/6
dr, dp = 0.25, np.pi/12
ph_range = np.linspace(p0, p0+dp, 30)
xouter = (r0+dr)*np.cos(ph_range)
youter = (r0+dr)*np.sin(ph_range)
xinner = r0*np.cos(ph_range[::-1])
yinner = r0*np.sin(ph_range[::-1])
poly_x = np.concatenate([xinner, xouter])
poly_y = np.concatenate([yinner, youter])
ax.fill(poly_x, poly_y, color=CORAL, alpha=0.4)
ax.plot(poly_x, poly_y, color=CORAL, lw=2)
area_polar = r0 * dr * dp
ax.text(1.15*np.cos(p0+dp/2), 1.15*np.sin(p0+dp/2),
        f'δA = ρ·δρ·δφ\n= {area_polar:.4f}',
        ha='center', va='center', fontsize=10, fontweight='bold')
ax.set_xlim(-0.1, 2.1); ax.set_ylim(-0.1, 2.1)
ax.set_aspect('equal'); ax.set_xlabel('x'); ax.set_ylabel('y')
ax.set_title('Polar grid (J = ρ — outer cells are larger!)')

plt.suptitle('Ch 2: Jacobian — How Coordinate Changes Distort Area', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### 2.2 Surface integral — flux through a spherical cap

In [ ]:
# ── Ch2: Flux through hemispherical cap ──────────────────────────────────────
# F = (-x, 0, z),  cap: sphere radius R, 0 <= theta <= alpha
# Exact: pi * R^3 * cos(alpha) * sin^2(alpha)
R     = 1.5    # sphere radius
alpha = np.pi/3  # cap angle  (try np.pi/2 for hemisphere)

theta = np.linspace(0, alpha, 30)
phi   = np.linspace(0, 2*np.pi, 50)
TH, PH = np.meshgrid(theta, phi)

Xs = R * np.sin(TH) * np.cos(PH)
Ys = R * np.sin(TH) * np.sin(PH)
Zs = R * np.cos(TH)

# Sample flux arrows
N_arr = 6
th_s = np.linspace(0.15, alpha-0.05, N_arr)
ph_s = np.linspace(0, 2*np.pi, N_arr, endpoint=False)
TH_s, PH_s = np.meshgrid(th_s, ph_s)
Xa = R * np.sin(TH_s) * np.cos(PH_s)
Ya = R * np.sin(TH_s) * np.sin(PH_s)
Za = R * np.cos(TH_s)
# F = (-x, 0, z)  then dot with outward normal er
# F·er = -x*sin(th)*cos(ph) + z*cos(th)
Fdotn = (-Xa * np.sin(TH_s)*np.cos(PH_s) + Za * np.cos(TH_s))

exact = np.pi * R**3 * np.cos(alpha) * np.sin(alpha)**2

fig = plt.figure(figsize=(13, 5))
ax1 = fig.add_subplot(1, 2, 1, projection='3d')
surf = ax1.plot_surface(Xs, Ys, Zs, alpha=0.35, color=TEAL, edgecolor='none')
# colour by F·n
ax1.quiver(Xa, Ya, Za, -Xa*0.2, np.zeros_like(Xa), Za*0.2,
           color=CORAL, length=0.25, normalize=False, alpha=0.8)
ax1.set_xlabel('x'); ax1.set_ylabel('y'); ax1.set_zlabel('z')
ax1.set_title(f'Spherical cap  (α = {np.degrees(alpha):.0f}°, R = {R})')

# Flux vs alpha
ax2 = fig.add_subplot(1, 2, 2)
alphas = np.linspace(0, np.pi, 200)
flux_a = np.pi * R**3 * np.cos(alphas) * np.sin(alphas)**2
ax2.plot(np.degrees(alphas), flux_a, color=TEAL, lw=2)
ax2.axvline(np.degrees(alpha), color=CORAL, ls='--', lw=1.5,
            label=f'α = {np.degrees(alpha):.0f}°\nflux = {exact:.4f}')
ax2.axhline(0, color=GRAY, lw=0.8)
ax2.set_xlabel('cap angle α (degrees)')
ax2.set_ylabel('flux  πR³ cos α sin²α')
ax2.set_title('Flux vs cap angle')
ax2.legend()
ax2.annotate('max flux', xy=(np.degrees(np.arctan(np.sqrt(2))), np.pi*R**3*2/(3*np.sqrt(3))),
             xytext=(80, 1.5), arrowprops=dict(arrowstyle='->', color=GRAY),
             fontsize=9, color=GRAY)

plt.suptitle('Ch 2: Flux of F = (−x, 0, z) through Spherical Cap', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()
print(f'Exact flux = π·R³·cos α·sin²α = {exact:.6f}')

---
## Chapter 3 — Grad, Div and Curl
### 3.1 Visualising grad, div, curl on three canonical fields

In [ ]:
# ── Ch3: Grad / Div / Curl on three fields ───────────────────────────────────
x, y = np.meshgrid(np.linspace(-2.5, 2.5, 20), np.linspace(-2.5, 2.5, 20))
xf, yf = np.meshgrid(np.linspace(-2.5, 2.5, 200), np.linspace(-2.5, 2.5, 200))

fields = [
    {'name': 'F = (x², 0)\ndiv = 2x, curl = 0',
     'U': x**2, 'V': np.zeros_like(x),
     'div': 2*xf, 'curl': np.zeros_like(xf), 'color': TEAL},
    {'name': 'F = (y, −x)\ndiv = 0, curl = −2',
     'U': y, 'V': -x,
     'div': np.zeros_like(xf), 'curl': -2*np.ones_like(xf), 'color': CORAL},
    {'name': 'F = ∇φ, φ = x²+y²\ndiv = 4, curl = 0',
     'U': 2*x, 'V': 2*y,
     'div': 4*np.ones_like(xf), 'curl': np.zeros_like(xf), 'color': PURPLE},
]

fig, axes = plt.subplots(3, 3, figsize=(14, 12))
titles_col = ['Vector field F', 'Divergence  ∇·F', 'Curl (z-component)  (∇×F)ᵤ']
for col, ttl in enumerate(titles_col):
    axes[0, col].set_title(ttl, fontsize=12, fontweight='bold')

for row, fd in enumerate(fields):
    ax = axes[row, 0]
    speed = np.sqrt(fd['U']**2 + fd['V']**2)
    speed[speed == 0] = 1e-10
    ax.quiver(x, y, fd['U']/speed, fd['V']/speed,
              speed, cmap='viridis', alpha=0.85, scale=22)
    ax.set_aspect('equal')
    ax.set_ylabel(fd['name'], fontsize=10, rotation=0, labelpad=160, va='center')
    ax.set_xlim(-2.5, 2.5); ax.set_ylim(-2.5, 2.5)

    for col, key in enumerate(['div', 'curl']):
        ax2 = axes[row, col+1]
        lim = max(abs(fd[key]).max(), 0.1)
        im = ax2.pcolormesh(xf, yf, fd[key], cmap='RdBu_r', vmin=-lim, vmax=lim, shading='auto')
        plt.colorbar(im, ax=ax2, shrink=0.8)
        ax2.set_aspect('equal')
        ax2.set_xlim(-2.5, 2.5); ax2.set_ylim(-2.5, 2.5)

plt.suptitle('Ch 3: Grad / Div / Curl — Three Canonical Fields', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

### 3.2 Conservative vs non-conservative fields

In [ ]:
# ── Ch3: Conservative vs non-conservative ─────────────────────────────────────
x, y = np.meshgrid(np.linspace(-2, 2, 18), np.linspace(-2, 2, 18))
xf, yf = np.meshgrid(np.linspace(-2, 2, 200), np.linspace(-2, 2, 200))

# Conservative: F = ∇φ, φ = x² + y²  → curl = 0
phi = xf**2 + yf**2
Uc, Vc = 2*x, 2*y

# Non-conservative: F = (-y, x) / (x²+y²)  → curl = 0 but ∮ ≠ 0!
r2 = x**2 + y**2 + 1e-6
Un, Vn = -y/r2, x/r2
r2f = xf**2 + yf**2 + 1e-6
phi_n = np.arctan2(yf, xf)  # multi-valued!

# Compute circulation around unit circle
t = np.linspace(0, 2*np.pi, 1000)
circ_x, circ_y = np.cos(t), np.sin(t)
Fx_on_C = -circ_y / (circ_x**2 + circ_y**2)
Fy_on_C =  circ_x / (circ_x**2 + circ_y**2)
dx_dt, dy_dt = -np.sin(t), np.cos(t)
circulation = np.trapz(Fx_on_C*dx_dt + Fy_on_C*dy_dt, t)

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Conservative field
ax = axes[0, 0]
ax.contourf(xf, yf, phi, levels=20, cmap='Blues', alpha=0.7)
ax.contour(xf, yf, phi, levels=10, colors='white', linewidths=0.6)
speed = np.sqrt(Uc**2 + Vc**2) + 1e-10
ax.quiver(x, y, Uc/speed, Vc/speed, color=TEAL, alpha=0.7, scale=20)
ax.set_title('Conservative: F = ∇φ, φ = x²+y²\n∮ F·dx = 0 for any closed curve')
ax.set_aspect('equal')

# Path-independence demo
ax = axes[0, 1]
t_path = np.linspace(0, 1, 300)
for path_y_exp, color, label in [(1, TEAL, 'C₁: (t,t)'), (2, CORAL, 'C₂: (t,t²)'), (3, PURPLE, 'C₃: (t,t³)')]:
    xp, yp = t_path, t_path**path_y_exp
    dxp, dyp = np.ones_like(t_path), path_y_exp * t_path**(path_y_exp-1)
    Fxp, Fyp = 2*xp, 2*yp
    val = np.trapz(Fxp*dxp + Fyp*dyp, t_path)
    ax.plot(xp, yp, color=color, lw=2, label=f'{label}  ∫={val:.3f}')
ax.scatter([0,1],[0,1], s=60, c='black', zorder=5)
ax.set_title('Conservative: all paths give same result\n(= φ(1,1)−φ(0,0) = 2)')
ax.legend(fontsize=9); ax.set_aspect('equal')
ax.set_xlim(-0.1, 1.2); ax.set_ylim(-0.1, 1.2)

# Non-conservative field
ax = axes[1, 0]
ax.contourf(xf, yf, phi_n, levels=20, cmap='RdBu', alpha=0.5)
mask = r2 > 0.05
speed_n = np.sqrt(Un**2 + Vn**2) + 1e-10
ax.quiver(x[mask], y[mask], (Un/speed_n)[mask], (Vn/speed_n)[mask],
          color=CORAL, alpha=0.7, scale=20)
circle = plt.Circle((0,0), 1, fill=False, color=BLUE, lw=2.5, ls='--')
ax.add_patch(circle)
ax.set_title(f'Non-conservative: F = (−y,x)/(x²+y²)\n∮ over unit circle = {circulation:.4f} ≈ 2π')
ax.set_aspect('equal')
ax.set_xlim(-2,2); ax.set_ylim(-2,2)

# Running circulation
ax = axes[1, 1]
running_circ = np.array([np.trapz((Fx_on_C*dx_dt + Fy_on_C*dy_dt)[:k+1], t[:k+1]) for k in range(0, len(t), 10)])
ax.plot(t[::10] * 180/np.pi, running_circ, color=CORAL, lw=2)
ax.axhline(2*np.pi, color=GRAY, ls='--', lw=1.5, label=f'2π = {2*np.pi:.4f}')
ax.set_xlabel('angle φ (degrees)')
ax.set_ylabel('∮ F·dx  (running total)')
ax.set_title(f'Circulation = {circulation:.5f} ≈ 2π\n(even though ∇×F = 0 everywhere except origin!)')
ax.legend()

plt.suptitle('Ch 3: Conservative vs Non-Conservative Fields', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Chapter 4 — Integral Theorems
### 4.1 Divergence theorem: volume integral = surface integral

In [ ]:
# ── Ch4: Divergence theorem — numerical verification ─────────────────────────
# F = (x³, y³, z³),  ∇·F = 3(x²+y²+z²) = 3r²
# Volume integral over ball of radius R:  ∫_V 3r² dV = (12π/5) R^5
# Surface integral over sphere:           ∮ F·dS = (12π/5) R^5  (same!)

R_values = np.linspace(0.3, 2.0, 20)
theory   = 12 * np.pi / 5 * R_values**5

# Numerical: volume integral in spherical coords
def vol_integral(R, N=60):
    r   = np.linspace(0, R, N)
    th  = np.linspace(0, np.pi, N)
    ph  = np.linspace(0, 2*np.pi, N)
    dr  = r[1]-r[0]; dth = th[1]-th[0]; dph = ph[1]-ph[0]
    RR, TH, PH = np.meshgrid(r, th, ph, indexing='ij')
    integrand = 3 * RR**2 * RR**2 * np.sin(TH)  # 3r² × Jacobian r²sinθ
    return integrand.sum() * dr * dth * dph

# Numerical: surface integral
def surf_integral(R, N=60):
    th  = np.linspace(0, np.pi, N)
    ph  = np.linspace(0, 2*np.pi, N)
    TH, PH = np.meshgrid(th, ph)
    dth = th[1]-th[0]; dph = ph[1]-ph[0]
    # F·n̂ on sphere = (x³,y³,z³)·(sinθcosφ, sinθsinφ, cosθ)
    # = R³(sin⁴θcos⁴φ + sin⁴θsin⁴φ + cos⁴θ)  ... actually F·r̂ × R^2sinθ
    # F = R³(sin³θcos³φ, sin³θsin³φ, cos³θ)
    # n̂ = (sinθcosφ, sinθsinφ, cosθ)
    Fdotn = R**3 * (np.sin(TH)**4 * np.cos(PH)**4
                  + np.sin(TH)**4 * np.sin(PH)**4
                  + np.cos(TH)**4)
    return (Fdotn * R**2 * np.sin(TH)).sum() * dth * dph

vol_num  = np.array([vol_integral(R) for R in R_values])
surf_num = np.array([surf_integral(R) for R in R_values])

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

ax = axes[0]
ax.plot(R_values, theory,   color=GRAY,   lw=3, ls='--', label='Theory: (12π/5)R⁵')
ax.plot(R_values, vol_num,  color=TEAL,   lw=2, marker='o', ms=4, label='Numerical: ∫_V ∇·F dV')
ax.plot(R_values, surf_num, color=CORAL,  lw=2, marker='s', ms=4, label='Numerical: ∮_S F·dS')
ax.set_xlabel('Radius R')
ax.set_ylabel('Integral value')
ax.set_title('Both sides of Gauss theorem agree')
ax.legend(fontsize=9)

ax = axes[1]
err_vol  = np.abs(vol_num  - theory) / theory * 100
err_surf = np.abs(surf_num - theory) / theory * 100
ax.semilogy(R_values, err_vol,  color=TEAL,  lw=2, label='Volume integral error %')
ax.semilogy(R_values, err_surf, color=CORAL, lw=2, label='Surface integral error %')
ax.set_xlabel('Radius R')
ax.set_ylabel('Relative error (%)')
ax.set_title('Numerical errors (from grid discretisation)')
ax.legend(fontsize=9)

plt.suptitle('Ch 4: Divergence Theorem — F = (x³,y³,z³), ∇·F = 3r²', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Chapter 5 — Poisson and Laplace Equations
### 5.1 Laplace equation: harmonic functions and equipotential lines

In [ ]:
# ── Ch5: Laplace equation solutions ──────────────────────────────────────────
x, y = np.meshgrid(np.linspace(-2, 2, 300), np.linspace(-2, 2, 300))
r    = np.sqrt(x**2 + y**2) + 1e-8

# Several harmonic functions (∇²φ = 0 for each)
solutions = [
    {'phi': x,               'name': 'φ = x   (l=1, m=0)'},
    {'phi': x**2 - y**2,     'name': 'φ = x²−y²   (l=2, m=0)'},
    {'phi': x*y,             'name': 'φ = xy   (l=2, m=2)'},
    {'phi': np.log(r),       'name': 'φ = ln r   (2D Green)'},
]

fig, axes = plt.subplots(2, 4, figsize=(16, 7))

for col, sol in enumerate(solutions):
    phi = sol['phi']
    # Check Laplacian numerically
    d2x = np.gradient(np.gradient(phi, axis=1), axis=1)
    d2y = np.gradient(np.gradient(phi, axis=0), axis=0)
    laplacian = d2x + d2y

    ax = axes[0, col]
    vmax = np.percentile(np.abs(phi), 95)
    im = ax.contourf(x, y, phi, levels=25, cmap='RdBu_r', vmin=-vmax, vmax=vmax)
    ax.contour(x, y, phi, levels=15, colors='white', linewidths=0.5, alpha=0.7)
    plt.colorbar(im, ax=ax, shrink=0.8)
    # Electric field = -∇φ
    xc, yc = np.meshgrid(np.linspace(-1.8,1.8,10), np.linspace(-1.8,1.8,10))
    rc = np.sqrt(xc**2+yc**2)+1e-8
    ax.set_title(sol['name'], fontsize=10)
    ax.set_aspect('equal')

    ax2 = axes[1, col]
    lmax = np.percentile(np.abs(laplacian[10:-10,10:-10]), 95)
    im2 = ax2.pcolormesh(x, y, laplacian, cmap='RdBu_r',
                         vmin=-max(lmax, 0.01), vmax=max(lmax, 0.01), shading='auto')
    plt.colorbar(im2, ax=ax2, shrink=0.8)
    ax2.set_title(f'∇²φ  (≈0 everywhere)', fontsize=10)
    ax2.set_aspect('equal')

axes[0,0].set_ylabel('φ (potential)', fontsize=11, fontweight='bold')
axes[1,0].set_ylabel('∇²φ (Laplacian)', fontsize=11, fontweight='bold')

plt.suptitle('Ch 5: Harmonic Functions — Solutions of ∇²φ = 0', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### 5.2 Poisson equation: point charge potential  ∇²φ = −ρ/ε₀

In [ ]:
# ── Ch5: Poisson — point charges ─────────────────────────────────────────────
# PARAMETERS
charges = [  # (x0, y0, q)
    ( 0.8,  0.0,  1.0),
    (-0.8,  0.0, -1.0),
]

x, y = np.meshgrid(np.linspace(-2.5, 2.5, 300), np.linspace(-2.5, 2.5, 300))
phi = np.zeros_like(x)
Ex  = np.zeros_like(x)
Ey  = np.zeros_like(x)

for x0, y0, q in charges:
    r   = np.sqrt((x-x0)**2 + (y-y0)**2) + 1e-6
    phi += q / r  # G = −1/(4π r) in 3D → 1/r in 2D analogy
    Ex  += -q * (x-x0) / r**3
    Ey  += -q * (y-y0) / r**3

# Clip for display
phi_display = np.clip(phi, -5, 5)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
im = ax.contourf(x, y, phi_display, levels=40, cmap='RdBu_r')
ax.contour(x, y, phi_display, levels=20, colors='white', linewidths=0.5, alpha=0.6)
plt.colorbar(im, ax=ax, label='potential φ')
for x0, y0, q in charges:
    color = CORAL if q > 0 else BLUE
    symbol = '+' if q > 0 else '−'
    ax.scatter(x0, y0, s=200, c=color, zorder=5, edgecolors='white', lw=2)
    ax.text(x0, y0+0.18, symbol, ha='center', fontsize=16, fontweight='bold',
            color='white')
ax.set_xlim(-2.5, 2.5); ax.set_ylim(-2.5, 2.5)
ax.set_aspect('equal')
ax.set_title('Potential φ — equipotential lines')

ax = axes[1]
ax.streamplot(x, y, Ex, Ey, color=np.sqrt(Ex**2+Ey**2),
              cmap='viridis', linewidth=0.8, density=1.2, arrowsize=1.2)
for x0, y0, q in charges:
    color = CORAL if q > 0 else BLUE
    symbol = '+' if q > 0 else '−'
    ax.scatter(x0, y0, s=200, c=color, zorder=5, edgecolors='white', lw=2)
    ax.text(x0, y0+0.18, symbol, ha='center', fontsize=16, fontweight='bold', color='white')
ax.set_xlim(-2.5, 2.5); ax.set_ylim(-2.5, 2.5)
ax.set_aspect('equal')
ax.set_title('Electric field E = −∇φ')

plt.suptitle('Ch 5: Poisson Equation — Dipole: ∇²φ = ρ,  φ = Σ qᵢ/rᵢ', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()
print('Try changing the charges list above — add more charges, flip signs, move them around!')

---
## Chapter 6 — Tensors and Index Notation
### 6.1 Levi-Civita symbol: verifying cross product and BAC-CAB

In [ ]:
# ── Ch6: Index notation — Levi-Civita verification ───────────────────────────
import itertools

# Build the ε tensor
eps = np.zeros((3, 3, 3))
for i, j, k in itertools.permutations([0,1,2]):
    perm = [i, j, k]
    inv  = sum(1 for a in range(3) for b in range(a+1, 3) if perm[a] > perm[b])
    eps[i, j, k] = (-1)**inv

# Test vectors
np.random.seed(42)
a = np.array([1.0, 2.0, 3.0])
b = np.array([4.0,-1.0, 2.0])
c = np.array([0.0, 3.0,-1.0])

# Cross product via ε: (a×b)_k = ε_ijk a_i b_j
axb_index = np.einsum('ijk,i,j->k', eps, a, b)
axb_numpy = np.cross(a, b)

# BAC-CAB:  a×(b×c) = b(a·c) − c(a·b)
bxc  = np.cross(b, c)
lhs  = np.cross(a, bxc)
rhs  = b * np.dot(a, c) - c * np.dot(a, b)

# ε–δ identity: ε_ijk ε_ilm = δ_jl δ_km − δ_jm δ_kl
lhs_ed = np.einsum('ijk,ilm->jklm', eps, eps)
delta  = np.eye(3)
rhs_ed = np.einsum('jl,km->jklm', delta, delta) - np.einsum('jm,kl->jklm', delta, delta)

print('='*60)
print('LEVI-CIVITA SYMBOL VERIFICATION')
print('='*60)
print(f'\nVectors:')
print(f'  a = {a}')
print(f'  b = {b}')
print(f'  c = {c}')

print(f'\n1. Cross product   (a × b):')
print(f'   Index notation:  {axb_index}')
print(f'   numpy.cross:     {axb_numpy}')
print(f'   Match: {np.allclose(axb_index, axb_numpy)} ✓')

print(f'\n2. BAC-CAB rule   a × (b × c) = b(a·c) − c(a·b):')
print(f'   LHS: {lhs}')
print(f'   RHS: {rhs}')
print(f'   Match: {np.allclose(lhs, rhs)} ✓')

print(f'\n3. ε–δ identity   ε_ijk ε_ilm = δ_jl δ_km − δ_jm δ_kl:')
print(f'   Max absolute error: {np.max(np.abs(lhs_ed - rhs_ed)):.2e} ✓')

# Visualise the ε tensor as three 3×3 slices
fig, axes = plt.subplots(1, 3, figsize=(10, 3))
for k in range(3):
    im = axes[k].imshow(eps[:,:,k], cmap='RdBu_r', vmin=-1, vmax=1, aspect='equal')
    axes[k].set_title(f'ε[:, :, {k}]  (k={k})')
    axes[k].set_xticks([0,1,2]); axes[k].set_xticklabels(['i=0','i=1','i=2'])
    axes[k].set_yticks([0,1,2]); axes[k].set_yticklabels(['j=0','j=1','j=2'])
    for i in range(3):
        for j in range(3):
            axes[k].text(j, i, f'{eps[i,j,k]:.0f}', ha='center', va='center',
                         fontsize=14, fontweight='bold',
                         color='white' if abs(eps[i,j,k]) > 0.5 else GRAY)
    plt.colorbar(im, ax=axes[k], shrink=0.7)

plt.suptitle('Ch 6: Levi-Civita Symbol εᵢⱼₖ — Three 3×3 Slices', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Summary — What to explore next

Each cell above has **PARAMETERS** you can tweak:

| Cell | Parameter | What changes |
|---|---|---|
| Helix | `lam`, `turns`, `frame_t` | Speed, number of loops, where the Frenet frame appears |
| Line integral | `F(x)` components | Whether the field is conservative or not |
| Jacobian | grid spacing | Larger cells → larger Jacobian effect |
| Spherical cap | `R`, `alpha` | Radius and cap angle; watch the flux change sign |
| ∇ operators | field expressions | Swap in any vector field of your choice |
| Conservative | `charges` in φ | Path (in)dependence |
| Divergence theorem | `R_values` | Verify the theorem at different scales |
| Laplace | `solutions` list | Add your own harmonic functions |
| Poisson | `charges` list | Add more charges, flip signs, make a quadrupole |

**Good luck with the course!  — PSE Lab, SeoulTech**